In [ ]:
import requests
import json
import os
from dotenv import load_dotenv

# 1. Carrega as variáveis escondidas no arquivo .env
load_dotenv()

# 2. Pega a chave de forma segura
minha_chave = os.getenv("API_KEY_FOOTBALL")

# 3. Definindo a URL e Parâmetros
url = "https://v3.football.api-sports.io/fixtures"
parametros = {
    "league": "71",
    "season": "2023"
}

# 4. O seu crachá de acesso usando a variável segura
headers = {
    "x-apisports-key": minha_chave
}

# 5. Fazendo a requisição
resposta = requests.get(url, headers=headers, params=parametros)
dados = resposta.json()

# 6. Validando o que chegou
if 'results' in dados and dados['results'] > 0:
    print(f"Sucesso! Total de partidas retornadas pela API: {dados['results']}")
    primeiro_time = dados['response'][0]
    print(json.dumps(primeiro_time, indent=2))
else:
    print("Nenhum dado retornado ou erro na conexão. Verifique o retorno da API:")
    print(dados)

Sucesso! Total de times retornados pela API: 380
{
  "fixture": {
    "id": 1005651,
    "referee": "Paulo Cesar Zanovelli da Silva",
    "timezone": "UTC",
    "date": "2023-04-15T19:00:00+00:00",
    "timestamp": 1681585200,
    "periods": {
      "first": 1681585200,
      "second": 1681588800
    },
    "venue": {
      "id": 258,
      "name": "Allianz Parque",
      "city": "S\u00e3o Paulo, S\u00e3o Paulo"
    },
    "status": {
      "long": "Match Finished",
      "short": "FT",
      "elapsed": 90,
      "extra": null
    }
  },
  "league": {
    "id": 71,
    "name": "Serie A",
    "country": "Brazil",
    "logo": "https://media.api-sports.io/football/leagues/71.png",
    "flag": "https://media.api-sports.io/flags/br.svg",
    "season": 2023,
    "round": "Regular Season - 1",
    "standings": true
  },
  "teams": {
    "home": {
      "id": 121,
      "name": "Palmeiras",
      "logo": "https://media.api-sports.io/football/teams/121.png",
      "winner": true
    },
    "awa

In [2]:
import pandas as pd

# 1. Isolando apenas a lista de jogos (que está dentro da chave 'response')
lista_jogos = dados['response']

estadios_unicos = {}

# Varrendo lista de jogos para extrair informações únicas de estádios
for jogo in lista_jogos:
    venue = jogo.get('fixture', {}).get('venue')
    if venue and venue.get('id'):
        estadio_id = venue.get('id')
        # Usamos um dicionário para garantir que não teremos estádios duplicados pelo ID
        estadios_unicos[estadio_id] = {
            "estadio_id": estadio_id,
            "nome_estadio": venue.get('name'),
            "cidade": venue.get('city')
        }

# Transformando o dicionário em DataFrame do Pandas
df_estadios = pd.DataFrame(list(estadios_unicos.values()))

# Exibindo as 5 primeiras linhas da nossa nova tabela limpa
print("Transformação concluída com sucesso! Veja a tabela:")
display(df_estadios.head()) # No Jupyter, o 'display' renderiza uma tabela visualmente mais bonita que o 'print'

Transformação concluída com sucesso! Veja a tabela:


,estadio_id,nome_estadio,cidade
0,258,Allianz Parque,"São Paulo, São Paulo"
1,206,Estádio Raimundo Sampaio,"Belo Horizonte, Minas Gerais"
2,218,Estádio Nilton Santos,Rio de Janeiro
3,10493,Arena da Baixada,"Curitiba, Paraná"
4,225,Estádio Governador Plácido Aderaldo Castelo,"Fortaleza, Ceará"


In [3]:
import sqlite3

# 1. Estabelecendo a conexão (o Python cria o arquivo do banco automaticamente na sua pasta)
conexao = sqlite3.connect('banco_brasileirao.db')

# 2. A Carga: Enviando o DataFrame do Pandas direto para a tabela do banco
# O parâmetro if_exists='replace' garante que, se rodarmos amanhã, ele atualiza a tabela sem duplicar tudo
df_estadios.to_sql(name='dim_estadios', con=conexao, if_exists='replace', index=False)

# 3. Fechando a conexão por segurança
conexao.close()

print("Carga concluída com sucesso! Os dados estão blindados no banco local.")

Carga concluída com sucesso! Os dados estão blindados no banco local.
